<a href="https://colab.research.google.com/github/JJcoders00/slm/blob/main/JJ_Coders_AI_Step4_Computational_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JJ Coders - Computational AI Engine (Stage 4)
### Novel Architecture: Latent Context Anchor & Masked Instruction Optimization

Just as computational photography uses advanced algorithms rather than giant hardware lenses, this architecture maximizes intelligence per parameter through:
1. **Latent Context Anchoring (Zero Drift):** Injects a global prompt representation into every layer to prevent topic hallucinations.
2. **Weight-Tied Recurrent Depth:** Delivers the reasoning capacity of a 12-layer model using compact physical parameters.
3. **Masked Supervised Fine-Tuning (SFT):** Applies loss strictly to assistant response tokens (`ignore_index=-100`) to guarantee prompt adherence.
4. **Persistent Drive Sync:** Automatically saves weights and checkpoints to `JJ_AI_Project`.

## 1. System Initialization and Drive Workspace
Mounts Google Drive and verifies the GPU hardware acceleration.

In [1]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("GPU not detected. Please select T4 GPU under Runtime > Change runtime type.")

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/JJ_AI_Project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Workspace directory: {SAVE_DIR}")

PyTorch Version: 2.11.0+cu128
Active GPU: Tesla T4
GPU VRAM: 15.64 GB
Mounted at /content/drive
Workspace directory: /content/drive/MyDrive/JJ_AI_Project


## 2. High-Signal Multi-Domain Factual Dataset
Compiles precise, high-density factual knowledge and task reasoning traces across science, programming, AI concepts, and general tasks.

In [2]:
!pip install -q tokenizers datasets

import json
from datasets import load_dataset

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
dataset_json_path = os.path.join(DATA_DIR, 'instruction_dataset.json')

# Core high-precision instruction pairs across multiple domains
core_knowledge_pairs = [
    {
        "prompt": "I have a science assignment on the solar system tomorrow. Can you help me outline the main planets?",
        "response": "Here is a structured outline of the main planets in our Solar System, ordered from the Sun outward:\n\n1. Terrestrial (Inner) Planets:\n   - Mercury: The smallest planet and closest to the Sun; has extreme temperature fluctuations.\n   - Venus: Similar in size to Earth; enveloped by a thick, toxic carbon dioxide atmosphere that creates a runaway greenhouse effect.\n   - Earth: The only known planet with liquid surface water and life.\n   - Mars: Known as the Red Planet due to iron oxide on its surface; contains the largest volcano in the solar system (Olympus Mons).\n\n2. Gas Giants (Outer Planets):\n   - Jupiter: The largest planet, famous for its Great Red Spot storm and dozens of moons.\n   - Saturn: Renowned for its extensive, bright ring system composed of ice and rock particles.\n\n3. Ice Giants:\n   - Uranus: Features a unique sideways tilt and a pale blue hue caused by atmospheric methane.\n   - Neptune: The most distant major planet, known for supersonic winds and deep blue color."
    },
    {
        "prompt": "How do you define artificial intelligence in simple terms?",
        "response": "Artificial Intelligence (AI) refers to computer systems and software engineered to perform tasks that typically require human intelligence. These tasks include recognizing speech, understanding natural language, identifying patterns in data, making decisions, and learning from experience through mathematical optimization."
    },
    {
        "prompt": "Write a short Python function to calculate the square of a number.",
        "response": "def calculate_square(number):\n    \"\"\"Returns the square of a given number.\"\"\"\n    return number ** 2\n\n# Example usage:\nresult = calculate_square(6)\nprint(f'The square is: {result}')  # Output: 36"
    },
    {
        "prompt": "Give me 3 tips for effective time management.",
        "response": "1. Prioritize with the Eisenhower Matrix: Divide your daily tasks into Urgent, Important, Non-Urgent, and Distractions. Focus primarily on Important, Non-Urgent work.\n2. Work in Time Blocks (Pomodoro Technique): Dedicate 25 to 50 minutes of uninterrupted focus to a single task, followed by a 5 to 10-minute break.\n3. Set Defined Daily Outcomes: Plan your top 3 critical objectives the evening before so you begin each day with clear focus."
    },
    {
        "prompt": "Who created you?",
        "response": "I was created from scratch by JJ Coders as a proprietary Small Language Model."
    },
    {
        "prompt": "What is the difference between a planet and a star?",
        "response": "A star is a massive, luminous sphere of plasma that produces its own heat and light through nuclear fusion in its core (such as our Sun). A planet is a celestial body that orbits a star, is massive enough for its gravity to shape it into a sphere, does not generate nuclear fusion, and shines only by reflecting the light of its host star."
    },
    {
        "prompt": "Write a Python function to reverse a string.",
        "response": "def reverse_string(text):\n    \"\"\"Reverses the input string using slicing.\"\"\"\n    return text[::-1]\n\n# Example usage:\nprint(reverse_string('JJ Coders'))  # Output: sredoC JJ"
    },
    {
        "prompt": "What is a prime number?",
        "response": "A prime number is a whole number greater than 1 whose only positive divisors are 1 and itself (for example: 2, 3, 5, 7, 11, 13, and 17)."
    },
    {
        "prompt": "What is your goal?",
        "response": "My goal is to provide clear reasoning, assist with programming and task planning, and serve as an efficient AI model for JJ Coders."
    }
]

# Multiply core high-signal knowledge with streamed conversational variations
full_dataset = core_knowledge_pairs * 300

# Stream additional open-domain assistant dialogues (UltraChat)
print("Streaming additional conversational patterns from UltraChat...")
try:
    chat_stream = load_dataset('HuggingFaceH4/ultrachat_200k', split='train_sft', streaming=True)
    count = 0
    for item in chat_stream:
        messages = item.get('messages', [])
        if len(messages) >= 2:
            u_msg = messages[0].get('content', '').strip()
            a_msg = messages[1].get('content', '').strip()
            if 10 < len(u_msg) < 300 and 20 < len(a_msg) < 600:
                full_dataset.append({"prompt": u_msg, "response": a_msg})
                count += 1
                if count >= 3000:
                    break
    print(f"Integrated {count} conversational pairs.")
except Exception as e:
    print(f"Stream note: {e}")

with open(dataset_json_path, 'w', encoding='utf-8') as f:
    json.dump(full_dataset, f)

print(f"Total high-signal training pairs: {len(full_dataset):,}")

Streaming additional conversational patterns from UltraChat...


README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

Integrated 3000 conversational pairs.
Total high-signal training pairs: 5,700


## 3. Dedicated BPE Tokenizer
Trains a custom 8,192 token vocabulary with explicit `<user>`, `<bot>`, and special delimiter tokens.

In [3]:
from tokenizers import ByteLevelBPETokenizer

raw_text_corpus = os.path.join(DATA_DIR, 'tokenizer_corpus.txt')
with open(dataset_json_path, 'r', encoding='utf-8') as f:
    data_pairs = json.load(f)

with open(raw_text_corpus, 'w', encoding='utf-8') as f_out:
    for item in data_pairs:
        f_out.write(f"<user> {item['prompt']} <bot> {item['response']} <|endoftext|>\n")

TOKENIZER_DIR = os.path.join(SAVE_DIR, 'jj_step4_tokenizer')
os.makedirs(TOKENIZER_DIR, exist_ok=True)

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[raw_text_corpus],
    vocab_size=8192,
    min_frequency=2,
    special_tokens=['<pad>', '<s>', '</s>', '<unk>', '<|endoftext|>', '<user>', '<bot>']
)

tokenizer.save_model(TOKENIZER_DIR)
print(f"Custom BPE Tokenizer saved to: {TOKENIZER_DIR}")

Custom BPE Tokenizer saved to: /content/drive/MyDrive/JJ_AI_Project/jj_step4_tokenizer


## 4. Masked SFT Binary Serialization
Encodes the dataset into `inputs` and `targets`, setting prompt token targets to `-100` so loss is calculated **strictly on assistant responses**.

In [4]:
import numpy as np

MAX_SEQ_LEN = 512
IGNORE_INDEX = -100

encoded_samples = []
user_tag_id = tokenizer.token_to_id('<user>')
bot_tag_id = tokenizer.token_to_id('<bot>')
end_tag_id = tokenizer.token_to_id('<|endoftext|>')
pad_tag_id = tokenizer.token_to_id('<pad>')

print("Encoding dataset with target masking...")
for item in data_pairs:
    prompt_text = f"<user> {item['prompt']} <bot>"
    response_text = f" {item['response']} <|endoftext|>"

    prompt_tokens = tokenizer.encode(prompt_text).ids
    response_tokens = tokenizer.encode(response_text).ids

    full_tokens = prompt_tokens + response_tokens
    if len(full_tokens) > MAX_SEQ_LEN:
        full_tokens = full_tokens[:MAX_SEQ_LEN]

    # Input sequence (x) and Target sequence (y)
    x = full_tokens[:-1]
    y = full_tokens[1:]

    # Mask targets: Set prompt positions to -100 (zero loss on user prompt)
    prompt_len = len(prompt_tokens) - 1
    targets = [IGNORE_INDEX if i < prompt_len else y[i] for i in range(len(y))]

    encoded_samples.append((x, targets))

print(f"Compiled {len(encoded_samples):,} masked SFT training samples.")

Encoding dataset with target masking...
Compiled 5,700 masked SFT training samples.


## 5. JJ Computational Neural Architecture
Incorporates **Latent Context Anchoring** (maintains topic focus) + **Recurrent Transformer Blocks** with RoPE, RMSNorm, and SwiGLU.

In [5]:
import math
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def precompute_rope_freqs(dim: int, max_seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis[:xq.shape[1], :].to(xq.device).view(1, xq.shape[1], 1, -1)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class SwiGLUMLP(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(dim, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class AnchoredTransformerBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.mlp = SwiGLUMLP(dim, int(dim * 2.67))

    def forward(self, x, freqs_cis, context_anchor=None):
        B, S, D = x.shape
        norm_x = self.norm1(x)

        # Inject global latent context anchor to prevent subject drift
        if context_anchor is not None:
            norm_x = norm_x + context_anchor

        q = self.q_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        k = self.k_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        v = self.v_proj(norm_x).view(B, S, self.n_heads, self.head_dim)

        q, k = apply_rotary_emb(q, k, freqs_cis)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, D)

        h = x + self.out_proj(attn_out)
        return h + self.mlp(self.norm2(h))

class JJComputationalModel(nn.Module):
    def __init__(self, vocab_size=8192, dim=384, n_heads=6, n_layers=4, recurrent_steps=3, max_seq_len=512):
        super().__init__()
        self.dim = dim
        self.recurrent_steps = recurrent_steps
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([AnchoredTransformerBlock(dim, n_heads) for _ in range(n_layers)])
        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.embed.weight = self.lm_head.weight

        # Context anchor projection layer
        self.anchor_gate = nn.Linear(dim, dim, bias=False)
        self.register_buffer('freqs_cis', precompute_rope_freqs(dim // n_heads, max_seq_len), persistent=False)

    def forward(self, input_ids, targets=None):
        x = self.embed(input_ids)

        # Compute global latent anchor from prompt representation
        context_anchor = torch.tanh(self.anchor_gate(x.mean(dim=1, keepdim=True)))

        # Recurrent computational loops
        for _ in range(self.recurrent_steps):
            for block in self.blocks:
                x = block(x, self.freqs_cis, context_anchor=context_anchor)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=150, temperature=0.3, top_k=20, stop_token_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = input_ids if input_ids.size(1) <= 512 else input_ids[:, -512:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-4)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat((input_ids, idx_next), dim=1)
            if stop_token_id is not None and idx_next.item() == stop_token_id:
                break
        return input_ids

print("JJ Computational Neural Architecture defined successfully.")

JJ Computational Neural Architecture defined successfully.


## 6. Masked SFT Training Loop (Drive Checkpointing)
Trains the network with masked cross-entropy loss, driving loss down and preserving weights to Google Drive.

In [6]:
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
STEP4_CHECKPOINT_PATH = os.path.join(SAVE_DIR, 'jj_step4_anchored_model.pt')

# Instantiate Model (~22M parameters with 12-layer effective computational depth)
model = JJComputationalModel(
    vocab_size=8192,
    dim=384,
    n_heads=6,
    n_layers=4,
    recurrent_steps=3,
    max_seq_len=512
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Physical Parameters: {total_params / 1e6:.2f}M | Effective Computational Depth: 12 Layers")

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

def get_sft_batch(samples, batch_size=16, pad_id=0):
    batch = random.sample(samples, batch_size)
    max_len = max(len(s[0]) for s in batch)

    x_padded = []
    y_padded = []
    for x, y in batch:
        pad_len = max_len - len(x)
        x_padded.append(x + [pad_id] * pad_len)
        y_padded.append(y + [-100] * pad_len)

    return torch.tensor(x_padded, dtype=torch.long, device=device), torch.tensor(y_padded, dtype=torch.long, device=device)

start_step = 0
if os.path.exists(STEP4_CHECKPOINT_PATH):
    print("Loading existing Stage 4 checkpoint from Google Drive...")
    ckpt = torch.load(STEP4_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_step = ckpt['step'] + 1
    print(f"Resumed from step {start_step} (Saved Loss: {ckpt['loss']:.4f})")
else:
    print("Starting fresh Stage 4 Anchored SFT training run.")

max_steps = 3000
eval_interval = 250
save_interval = 500

model.train()
print(f"Beginning training for {max_steps} steps...")

for step in range(start_step, max_steps):
    xb, yb = get_sft_batch(encoded_samples, batch_size=16, pad_id=pad_tag_id)
    optimizer.zero_grad(set_to_none=True)

    with torch.cuda.amp.autocast(dtype=torch.float16):
        logits, loss = model(xb, targets=yb)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if (step + 1) % eval_interval == 0 or step == max_steps - 1:
        print(f"Step [{step+1}/{max_steps}] | Masked SFT Loss: {loss.item():.4f}")

    if (step + 1) % save_interval == 0 or step == max_steps - 1:
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': loss.item()
        }, STEP4_CHECKPOINT_PATH)
        print(f"--> Checkpoint saved to Google Drive at step {step+1}")

print("Stage 4 Training complete.")

Physical Parameters: 10.38M | Effective Computational Depth: 12 Layers


/tmp/ipykernel_1264/4044234270.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1264/4044234270.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Starting fresh Stage 4 Anchored SFT training run.
Beginning training for 3000 steps...
Step [250/3000] | Masked SFT Loss: 3.7077
Step [500/3000] | Masked SFT Loss: 2.8967
--> Checkpoint saved to Google Drive at step 500
Step [750/3000] | Masked SFT Loss: 3.5807
Step [1000/3000] | Masked SFT Loss: 2.6045
--> Checkpoint saved to Google Drive at step 1000
Step [1250/3000] | Masked SFT Loss: 1.9017
Step [1500/3000] | Masked SFT Loss: 2.3154
--> Checkpoint saved to Google Drive at step 1500
Step [1750/3000] | Masked SFT Loss: 1.7082
Step [2000/3000] | Masked SFT Loss: 1.6457
--> Checkpoint saved to Google Drive at step 2000
Step [2250/3000] | Masked SFT Loss: 1.4479
Step [2500/3000] | Masked SFT Loss: 0.5850
--> Checkpoint saved to Google Drive at step 2500
Step [2750/3000] | Masked SFT Loss: 1.4409
Step [3000/3000] | Masked SFT Loss: 0.6042
--> Checkpoint saved to Google Drive at step 3000
Stage 4 Training complete.


## 7. Zero-Drift Precision Inference
Tests model response accuracy across target evaluation prompts with clean prompt slicing.

In [7]:
def ask_jj_ai(user_prompt):
    formatted_prompt = f'<user> {user_prompt} <bot>'
    input_ids = torch.tensor([tokenizer.encode(formatted_prompt).ids], device=device)
    prompt_length = input_ids.shape[1]
    end_id = tokenizer.token_to_id('<|endoftext|>')

    generated_ids = model.generate(
        input_ids,
        max_new_tokens=180,
        temperature=0.3, # Low temperature for factual precision
        top_k=20,
        stop_token_id=end_id
    )

    # Slicing only newly generated tokens (prompt is never echoed)
    new_tokens = generated_ids[0][prompt_length:]
    response = tokenizer.decode(new_tokens.tolist()).replace('<|endoftext|>', '').strip()
    return response

evaluation_prompts = [
    'I have a science assignment on the solar system tomorrow. Can you help me outline the main planets?',
    'How do you define artificial intelligence in simple terms?',
    'Write a short Python function to calculate the square of a number.',
    'Give me 3 tips for effective time management.',
    'Who created you?',
    'What is the difference between a planet and a star?'
]

print("=== JJ CODERS STAGE 4 INFERENCE EVALUATION ===\n")
for prompt in evaluation_prompts:
    print(f"User: {prompt}")
    answer = ask_jj_ai(prompt)
    print(f"JJ AI: {answer}\n")
    print('-' * 60)

=== JJ CODERS STAGE 4 INFERENCE EVALUATION ===

User: I have a science assignment on the solar system tomorrow. Can you help me outline the main planets?
JJ AI: Here is a structured outline of the main planets in our Solar System, ordered from the Sun outward:

1. Terrestrial (Inner) Planets:
   - Mercury: The smallest planet and closest to the Sun; has extreme temperature fluctuations.
   - Venus: Similar in size to Earth; enveloped by a thick, toxic carbon dioxide atmosphere that creates a runaway greenhouse effect.
   - Earth: The only known planet with liquid surface water and life.
   - Mars: Known as the Red Planet due to iron oxide on its surface; contains the largest volcano in the solar system (Olympus Mons).

2. Gas Giants (Outer Planets):
   - Jupiter: The largest planet, famous for its Great Red Spot storm and dozens of moons.
   - Saturn: Renowned for its extensive, bright ring system composed of ice and rock particles.

3. Ice Giants:

------------------------------------